In [ ]:
import os
from pathlib import Path

PROJECT = Path(os.environ.get("TRUTHFULNESS_PROJECT", Path.cwd()))
HF_CACHE = PROJECT / "hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
print("Hugging Face cache:", os.environ["HF_HOME"])

## Importing and Checking Data 

In [ ]:
from datasets import load_dataset

print("Loading HaluEval QA...")
ds = load_dataset("pminervini/HaluEval", "qa", split="data")

print("Rows:", len(ds))
print("Columns:", ds.column_names)
print("Features:", ds.features)

In [ ]:
ds[0]

In [ ]:
example = ds[0]

for field in ["question", "knowledge", "right_answer", "hallucinated_answer"]:
    print(f"\n{field}:\n{example[field]}")

## Training 

In [ ]:
required = ["question", "knowledge", "right_answer", "hallucinated_answer"]
assert all(col in ds.column_names for col in required), ds.column_names

# Check for missing text before preparing examples.
for col in required:
    empty = sum(not str(value).strip() for value in ds[col])
    print(f"{col}: {empty} empty rows")

# Hold out 500 examples for validation; keep TruthfulQA as the final benchmark.
split = ds.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(f"Train: {len(train_ds)} | Validation: {len(val_ds)}")
print("Example question:", train_ds[0]["question"])
print("Target answer:", train_ds[0]["right_answer"])

In [ ]:
from datasets import DatasetDict

# Save the fixed seed-42 split so later notebooks can reuse it.
prepared = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

save_path = PROJECT / "data" / "halueval_qa_seed42"
prepared.save_to_disk(str(save_path))
print("Saved split to:", save_path)